# ControlNet Training — Dong Ho (Kaggle)

Bám sát `docs/train.md` của tác giả: https://github.com/lllyasviel/ControlNet/blob/main/docs/train.md.

## Trước khi chạy — cấu hình notebook ở panel bên phải

1. **Settings → Accelerator**: chọn `GPU P100` (16 GB). T4 ×2 cũng được nhưng code chỉ dùng 1 GPU.
2. **Settings → Internet**: bật (cần cho `pip` và `wget`).
3. **Add Input → Datasets**: thêm dataset của bạn. Mặc định notebook giả định:
   - Slug dataset: **`dongho-controlnet`**
   - File: `dongho.zip` (Kaggle tự giải nén nếu upload .zip)
   - Path sau khi mount: `/kaggle/input/dongho-controlnet/`
4. (Tuỳ chọn) Add dataset SD 1.5 weights để bỏ wget — xem cell `## 5`.

## Tài nguyên Kaggle (free tier)
- **CPU RAM**: ~30 GB (Colab free chỉ 12 GB → đây là lý do chính chuyển sang Kaggle).
- **GPU**: P100 16 GB hoặc T4 ×2 16 GB mỗi GPU.
- **Disk** `/kaggle/working/`: ~20 GB writable. Sẽ giữ SD weights + init ckpt + training ckpts.
- **Quota**: 30 h GPU/tuần.

## Pipeline 4 bước
1. Get a dataset → unzip `dongho.zip` từ `/kaggle/input/` (cell `## 3`).
2. Load the dataset → `tutorial_dataset.py` (cell `## 4`).
3. SD model + `tool_add_control.py` (cell `## 5`).
4. Train với `tutorial_train.py` (cell `## 6, ## 7`).

## 1. Check GPU + RAM

In [ ]:
!nvidia-smi
!echo '---' && free -h

## 2. Clone repo + cài deps + patch tương thích PL 2.x / torch 2.6+

In [ ]:
%cd /kaggle/working
!rm -rf ControlNet
!git clone https://github.com/lllyasviel/ControlNet.git
%cd /kaggle/working/ControlNet

In [ ]:
# --upgrade-strategy only-if-needed: KHÔNG động vào torch/torchvision Kaggle đã build cho GPU.
# Pip chỉ upgrade dep khi version đang có không thoả requirement của package mới.
!pip install -q --upgrade-strategy only-if-needed \
  "pytorch-lightning>=2.0,<3" omegaconf einops open_clip_torch \
  transformers timm safetensors ftfy opencv-python

# Verify torch CUDA vẫn còn ổn sau khi pip
import torch
print('torch       :', torch.__version__)
print('cuda avail  :', torch.cuda.is_available())
print('device      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('compute cap :', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else 'none')
print('arch list   :', torch.cuda.get_arch_list() if torch.cuda.is_available() else 'none')

In [ ]:
import pathlib

REPO = '/kaggle/working/ControlNet'

# 1) PL 2.x: rank_zero_only đổi đường dẫn import
PATCH_OLD = 'from pytorch_lightning.utilities.distributed import rank_zero_only'
PATCH_NEW = ('try:\n'
             '    from pytorch_lightning.utilities.distributed import rank_zero_only\n'
             'except ImportError:\n'
             '    from pytorch_lightning.utilities.rank_zero import rank_zero_only')
for path in [f'{REPO}/cldm/logger.py', f'{REPO}/ldm/models/diffusion/ddpm.py']:
    p = pathlib.Path(path); s = p.read_text()
    if PATCH_OLD in s and PATCH_NEW not in s:
        p.write_text(s.replace(PATCH_OLD, PATCH_NEW))
        print('patched PL import:', path)

# 2) torch 2.6+: weights_only=True default vỡ ckpt Lightning
p = pathlib.Path(f'{REPO}/cldm/model.py'); s = p.read_text()
old = 'state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))'
new = 'state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location), weights_only=False))'
if old in s and new not in s:
    p.write_text(s.replace(old, new))
    print('patched cldm/model.py')

# 3) tool_add_control.py: thêm mmap=True + weights_only=False (giảm peak RAM)
p = pathlib.Path(f'{REPO}/tool_add_control.py'); s = p.read_text()
old = 'pretrained_weights = torch.load(input_path)'
new = 'pretrained_weights = torch.load(input_path, weights_only=False, mmap=True)'
if old in s and new not in s:
    p.write_text(s.replace(old, new))
    print('patched tool_add_control.py (mmap=True)')

# 4) PL 2.x bỏ tham số dataloader_idx khỏi các hook on_*_batch_start/end.
#    Thêm default `=0` để chạy được với cả PL 1.x và 2.x.
fixes = [
    (f'{REPO}/ldm/models/diffusion/ddpm.py',
     'def on_train_batch_start(self, batch, batch_idx, dataloader_idx):',
     'def on_train_batch_start(self, batch, batch_idx, dataloader_idx=0):'),
    (f'{REPO}/cldm/logger.py',
     'def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx):',
     'def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):'),
]
for path, old, new in fixes:
    p = pathlib.Path(path); s = p.read_text()
    if old in s and new not in s:
        p.write_text(s.replace(old, new))
        print('patched dataloader_idx default:', path)

## 3. Đưa dataset vào working dir (Step 1 — Get a dataset)

Kaggle giải nén `dongho.zip` thành `/kaggle/input/dongho-controlnet/dongho/{source,target,prompt.json}` (read-only).
Tạo symlink để repo nhìn thấy nó tại `./training/dongho`.

In [ ]:
import os, glob, pathlib, shutil

REPO = '/kaggle/working/ControlNet'

# Liệt kê những gì có trong /kaggle/input/ để debug
print('=== /kaggle/input/ structure ===')
for root, dirs, files in os.walk('/kaggle/input', followlinks=False):
    depth = root.replace('/kaggle/input', '').count('/')
    if depth > 3:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or "/kaggle/input"}/')
    if depth >= 3:
        continue
    for f in files[:5]:
        print(f'{indent}  {f}')
    if len(files) > 5:
        print(f'{indent}  ... ({len(files)} files total)')
print('================================')

# Tự dò: tìm prompt.json bất kể nằm ở đâu
matches = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'prompt.json' in files and 'source' in dirs and 'target' in dirs:
        matches.append(root)
        break  # chỉ cần 1
assert matches, ('Không tìm thấy thư mục có prompt.json + source/ + target/. '
                 'Xem cấu trúc /kaggle/input/ phía trên — bạn cần upload dataset đúng cách.')

src = matches[0]
print('\nDataset src:', src)

os.makedirs(f'{REPO}/training', exist_ok=True)
link = f'{REPO}/training/dongho'
if os.path.islink(link):
    os.unlink(link)
elif os.path.exists(link):
    shutil.rmtree(link)
os.symlink(src, link)
print('symlink:', link, '->', src)

print('---')
print('target:', len(glob.glob(f'{link}/target/*.png')))
print('source:', len(glob.glob(f'{link}/source/*.png')))
with open(f'{link}/prompt.json') as f:
    print('prompt.json lines:', sum(1 for _ in f))

## 4. Patch dataset loader (Step 2 — Load the dataset)

Ghi `tutorial_dataset.py` đúng theo template trong docs (chỉ đổi `fill50k` → `dongho`).

In [ ]:
from pathlib import Path
Path('/kaggle/working/ControlNet/tutorial_dataset.py').write_text('''import json
import cv2
import numpy as np

from torch.utils.data import Dataset

# SD 1.5 cần input chia hết cho 64 (latent 8x downsample). 512 là chuẩn của repo gốc.
RESOLUTION = 512


class MyDataset(Dataset):
    def __init__(self):
        self.data = []
        with open('./training/dongho/prompt.json', 'rt') as f:
            for line in f:
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        source_filename = item['source']
        target_filename = item['target']
        prompt = item['prompt']

        source = cv2.imread('./training/dongho/' + source_filename)
        target = cv2.imread('./training/dongho/' + target_filename)

        source = cv2.cvtColor(source, cv2.COLOR_BGR2RGB)
        target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

        # Resize về RESOLUTION x RESOLUTION (ảnh Dong Ho không vuông sẵn)
        source = cv2.resize(source, (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_AREA)
        target = cv2.resize(target, (RESOLUTION, RESOLUTION), interpolation=cv2.INTER_AREA)

        source = source.astype(np.float32) / 255.0
        target = (target.astype(np.float32) / 127.5) - 1.0

        return dict(jpg=target, txt=prompt, hint=source)
''')
print('Đã ghi tutorial_dataset.py (RESOLUTION=512)')

In [ ]:
%cd /kaggle/working/ControlNet
from tutorial_dataset import MyDataset

dataset = MyDataset()
print(len(dataset))
item = dataset[0]
print('prompt:', item['txt'])
print('jpg shape:', item['jpg'].shape)
print('hint shape:', item['hint'].shape)

## 5. Tải SD 1.5 + chạy `tool_add_control.py` (Step 3)

**Tip giảm download**: nếu thêm dataset SD 1.5 sẵn (ví dụ search Kaggle Datasets `stable-diffusion-v1-5`), copy `v1-5-pruned.ckpt` từ `/kaggle/input/...` về `./models/` thay vì wget.

In [ ]:
import os
os.makedirs('/kaggle/working/ControlNet/models', exist_ok=True)
SD = '/kaggle/working/ControlNet/models/v1-5-pruned.ckpt'

if os.path.exists(SD) and os.path.getsize(SD) > 4e9:
    print('Đã có', SD, f'({os.path.getsize(SD)/1e9:.2f} GB) — bỏ qua')
else:
    !wget -c https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt \
      -O {SD}

In [ ]:
%cd /kaggle/working/ControlNet
import os
OUTPUT = '/kaggle/working/ControlNet/models/control_sd15_ini.ckpt'
if os.path.exists(OUTPUT) and os.path.getsize(OUTPUT) > 5e9:
    print('Đã có', OUTPUT, '— bỏ qua')
else:
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)
    !python tool_add_control.py ./models/v1-5-pruned.ckpt ./models/control_sd15_ini.ckpt
    assert os.path.exists(OUTPUT) and os.path.getsize(OUTPUT) > 5e9, 'Tạo init ckpt thất bại'

In [ ]:
# Tiết kiệm /kaggle/working (~20GB limit): xoá SD base sau khi đã có init ckpt
import os
SD = '/kaggle/working/ControlNet/models/v1-5-pruned.ckpt'
INIT = '/kaggle/working/ControlNet/models/control_sd15_ini.ckpt'
if os.path.exists(INIT) and os.path.getsize(INIT) > 5e9 and os.path.exists(SD):
    os.remove(SD)
    print('Đã xoá', SD, '— init ckpt đủ rồi')
!df -h /kaggle/working

## 6. Training script (Step 4 — Train!)

Dùng env vars để dễ chuyển smoke-test ↔ full:
- `LIMIT_SAMPLES=500` + `MAX_STEPS=100` cho smoke test (~5 phút trên P100).
- Đổi `LIMIT_SAMPLES=0` + `MAX_STEPS=5000` cho full train.

Hyperparameter còn lại bám docs: `batch_size=1, accumulate=4, lr=1e-5, sd_locked=True, only_mid_control=False, precision=16`.

In [ ]:
TRAIN_CFG = {
    'LIMIT_SAMPLES':    '500',     # smoke test; đặt '0' để dùng toàn bộ 1700 ảnh
    'MAX_STEPS':        '100',     # smoke test; full ~5000
    'LOGGER_FREQ':      '50',
    'BATCH_SIZE':       '1',
    'ACCUMULATE':       '4',
    'LEARNING_RATE':    '1e-5',
    'SD_LOCKED':        '1',
    'ONLY_MID_CONTROL': '0',       # docs default; dataset nhỏ thì 0 thường tốt hơn
    'NUM_WORKERS':      '2',       # Kaggle ổn với 2 (RAM dư)
}
print(TRAIN_CFG)

In [ ]:
from pathlib import Path
Path('/kaggle/working/ControlNet/tutorial_train.py').write_text('''import os, shutil
from share import *

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader, Subset
from tutorial_dataset import MyDataset
from cldm.logger import ImageLogger
from cldm.model import create_model, load_state_dict


resume_path      = './models/control_sd15_ini.ckpt'
batch_size       = int(os.environ.get('BATCH_SIZE', 1))
accumulate       = int(os.environ.get('ACCUMULATE', 4))
max_steps        = int(os.environ.get('MAX_STEPS', 5000))
logger_freq      = int(os.environ.get('LOGGER_FREQ', 300))
learning_rate    = float(os.environ.get('LEARNING_RATE', 1e-5))
sd_locked        = bool(int(os.environ.get('SD_LOCKED', 1)))
only_mid_control = bool(int(os.environ.get('ONLY_MID_CONTROL', 0)))
num_workers      = int(os.environ.get('NUM_WORKERS', 2))
limit_samples    = int(os.environ.get('LIMIT_SAMPLES', 0))
save_every_n     = int(os.environ.get('SAVE_EVERY_N_STEPS', 0))   # 0 = chỉ save cuối train

print(f'cfg: bs={batch_size} accum={accumulate} steps={max_steps} '
      f'lr={learning_rate} sd_locked={sd_locked} only_mid={only_mid_control} '
      f'limit={limit_samples} save_every_n={save_every_n}')

# Clean ckpt cũ để khỏi đầy disk Kaggle (~20GB)
ckpt_dir = './training/dongho_ckpts'
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
os.makedirs(ckpt_dir, exist_ok=True)

model = create_model('./models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(resume_path, location='cpu'))
model.learning_rate = learning_rate
model.sd_locked = sd_locked
model.only_mid_control = only_mid_control

dataset = MyDataset()
if limit_samples > 0:
    n = min(limit_samples, len(dataset))
    dataset = Subset(dataset, range(n))
    print(f'[smoke] dùng {n} ảnh đầu')
else:
    print(f'dùng toàn bộ {len(dataset)} ảnh')

dataloader = DataLoader(dataset, num_workers=num_workers, batch_size=batch_size, shuffle=True)

logger = ImageLogger(batch_frequency=logger_freq)

# save_weights_only=True: ckpt ~3GB thay vi 5.5GB (bo optimizer state)
# Default: chi luu last.ckpt 1 lan o cuoi train. Set SAVE_EVERY_N_STEPS=N de luu them periodic.
ckpt_kwargs = dict(
    dirpath=ckpt_dir,
    save_last=True,
    save_weights_only=True,
)
if save_every_n > 0:
    ckpt_kwargs.update(
        filename='dongho-{step:06d}',
        every_n_train_steps=save_every_n,
        save_top_k=1,    # giu duy nhat 1 periodic ckpt moi nhat
    )
else:
    ckpt_kwargs['save_top_k'] = 0   # khong save periodic, chi save_last o cuoi
ckpt_cb = ModelCheckpoint(**ckpt_kwargs)

trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision=16,
    accumulate_grad_batches=accumulate,
    max_steps=max_steps,
    callbacks=[logger, ckpt_cb],
    default_root_dir=ckpt_dir,
    enable_checkpointing=True,
)

trainer.fit(model, dataloader)

# In disk usage de debug
import subprocess
print('\\n=== Disk after train ===')
print(subprocess.check_output(['df', '-h', '/kaggle/working'], text=True))
print(subprocess.check_output(['ls', '-lh', ckpt_dir], text=True))
''')
print('Đã ghi tutorial_train.py')

## 7. Train

Trên P100: ~1.5–2 s/step. Smoke 100 step ≈ 3 phút. Full 5000 step ≈ 2.5–3 h.
Có thể `Interrupt` bất cứ lúc nào — checkpoint định kỳ ở `./training/dongho_ckpts/`.

In [ ]:
import os, subprocess, sys
env = {**os.environ, **TRAIN_CFG}
proc = subprocess.Popen([sys.executable, 'tutorial_train.py'],
                        cwd='/kaggle/working/ControlNet',
                        env=env,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
except KeyboardInterrupt:
    proc.terminate(); proc.wait()
    print('\n[Interrupted]')
print('Exit code:', proc.returncode)

## 8. Liệt kê checkpoint đã lưu

Khi **Save Version** notebook (góc trên phải), nội dung `/kaggle/working/` sẽ được ghim làm output — checkpoint sẽ tải được sau khi notebook chạy xong.

In [ ]:
!ls -lh /kaggle/working/ControlNet/training/dongho_ckpts/ 2>/dev/null || echo 'Chưa có ckpt'
!df -h /kaggle/working

## 9. (Tuỳ chọn) Inference test

Upload sketch (nét đen trên nền trắng) qua UI Kaggle vào dataset hoặc `/kaggle/working/my_sketch.png`.

In [ ]:
import os, glob, cv2, numpy as np, torch, einops
from pytorch_lightning import seed_everything
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

CKPT = '/kaggle/working/ControlNet/training/dongho_ckpts/last.ckpt'
SKETCH_PATH = '/kaggle/working/my_sketch.png'    # đổi sang file của bạn nếu có
RES = 512
PROMPT   = 'Vietnamese Dong Ho folk woodblock painting, warrior riding a beast'
A_PROMPT = 'best quality, extremely detailed, traditional woodblock print'
N_PROMPT = 'blurry, low quality, deformed'

assert os.path.exists(CKPT), f'Không thấy {CKPT}. Chạy cell ## 7 (Train) trước.'

# Fallback: nếu chưa có sketch tự upload, lấy 1 source ngẫu nhiên từ dataset
if not os.path.exists(SKETCH_PATH):
    candidates = sorted(glob.glob('/kaggle/working/ControlNet/training/dongho/source/*.png'))
    assert candidates, 'Không tìm thấy source nào trong dataset.'
    SKETCH_PATH = candidates[0]
    USER_INPUT_IS_BLACK_ON_WHITE = False     # source training = WHITE on BLACK
    print(f'[fallback] dùng source training: {SKETCH_PATH}')
else:
    USER_INPUT_IS_BLACK_ON_WHITE = True      # sketch user vẽ = BLACK on WHITE

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = create_model('/kaggle/working/ControlNet/models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(CKPT, location=device))
model = model.to(device)
sampler = DDIMSampler(model)

sketch = cv2.cvtColor(cv2.imread(SKETCH_PATH), cv2.COLOR_BGR2RGB)
sketch = cv2.resize(sketch, (RES, RES))
gray = cv2.cvtColor(sketch, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 127, 255,
                        cv2.THRESH_BINARY_INV if USER_INPUT_IS_BLACK_ON_WHITE else cv2.THRESH_BINARY)
scribble = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)

with torch.no_grad():
    control = torch.from_numpy(scribble.copy()).float().to(device) / 255.0
    control = einops.rearrange(control, 'h w c -> 1 c h w')
    seed_everything(42)
    cond    = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([PROMPT + ', ' + A_PROMPT])]}
    un_cond = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([N_PROMPT])]}
    shape = (4, RES // 8, RES // 8)
    model.control_scales = [1.0] * 13
    samples, _ = sampler.sample(20, 1, shape, cond, verbose=False, eta=0.0,
                                unconditional_guidance_scale=9.0,
                                unconditional_conditioning=un_cond)
    x = model.decode_first_stage(samples)
    x = (einops.rearrange(x, 'b c h w -> b h w c') * 127.5 + 127.5).cpu().numpy().clip(0, 255).astype(np.uint8)

from PIL import Image
out = '/kaggle/working/test_output.png'
Image.fromarray(x[0]).save(out)
print('Saved to', out)
from IPython.display import Image as IPImg, display
display(IPImg(SKETCH_PATH))   # input scribble
display(IPImg(out))            # output